# 007 - JupyterLab Sidebar Chatbot 데모

JupyterLab **우측 사이드바**에 뜨는 챗봇입니다. **jupyter 서버를 재시작할 수 없는 환경**(회사 폐쇄망)을 위해, 설치부터 실행까지 **전부 노트북 셀 안에서** 끝나도록 만들었습니다.

| 단계 | 하는 일 | jupyter 재시작? |
|---|---|---|
| ① 프론트 설치 | 셀에서 `%pip install <wheel>` → 브라우저 새로고침 → 우측 💬 탭 등장 | ❌ 불필요 |
| ② 두뇌 시작 | 셀에서 `start_brain_server()` (localhost:8765) | ❌ 불필요 |
| ③ 대화 | 💬 탭에서 입력 | — |

> ⚠️ 전제: 브라우저의 `127.0.0.1` 과 이 커널이 **같은 기계**여야 합니다(로컬/컨테이너/SSH 포트포워딩). single-file `.py` 가 아니라 빌드가 필요한 익스텐션입니다 — 자세한 건 `README.md` 참고.

## ① 프론트엔드 설치 — 노트북 셀에서 (jupyter 재시작 불필요)

터미널을 못 여는 환경이라면 **셀에서 직접 `%pip install`** 하면 됩니다. `%pip` 매직은 지금 이 커널(=jupyter 서버와 같은 env)에 설치하므로, labextension 자산이 `share/jupyter/labextensions/` 에 들어갑니다. 폐쇄망에서는 반입한 `.whl` 경로를 지정하세요.

In [ ]:
# 빌드 산출물 dist/ 의 wheel 을 설치 (`pip wheel . -w dist` 결과). 폐쇄망에선 반입한 .whl 경로로 바꾸세요.
%pip install dist/jlab_sidebar_chatbot-0.1.0-py3-none-any.whl
# 설치 후 → 브라우저에서 JupyterLab 페이지를 '새로고침'(Cmd/Ctrl+R) 하면 우측에 💬 탭이 생깁니다.
# (서버 재시작 불필요: jupyter 는 페이지를 그릴 때마다 labextensions 폴더를 다시 스캔하기 때문)

## ② 두뇌 서버 시작 — 이 셀 한 줄

커널 안에서 localhost HTTP 서버(기본 포트 8765)를 백그라운드로 띄웁니다. 우측 💬 탭에서 보낸 메시지가 이 서버로 전달됩니다. (방금 `%pip install` 한 패키지는 이 커널에서 바로 import 됩니다 — 커널 재시작도 불필요.)

In [ ]:
from jlab_sidebar_chatbot import start_brain_server

start_brain_server()   # → http://127.0.0.1:8765 (백그라운드 스레드)
# 이제 오른쪽 사이드바의 💬 탭을 열고 대화해 보세요.
# 중지:  from jlab_sidebar_chatbot import stop_brain_server; stop_brain_server()

## ③ (옵션) 두뇌만 빠르게 체험 — 서버/탭 없이

사이드바 없이 두뇌(`jlab_sidebar_chatbot/llm.py`)만 바로 확인하고 싶을 때.

In [ ]:
from jlab_sidebar_chatbot import ChatBrain

brain = ChatBrain(system_prompt='너는 폐쇄망 데모 도우미야.')
sid = 'notebook-demo'
for line in ['안녕하세요', '이거 어떻게 쓰나요?', '고맙습니다']:
    ans = brain.send(sid, line)
    print(f'🙂 user: {line}')
    print(f'🤖 bot : {ans["content"]}\n')

## ④ 사내 LLM 으로 교체

`LLMAdapter` 를 구현해 `start_brain_server(brain=...)` 로 주입합니다.

```python
from jlab_sidebar_chatbot import start_brain_server, ChatBrain
from jlab_sidebar_chatbot.llm import LLMAdapter

class MyLocalLLM(LLMAdapter):
    def generate(self, messages):
        # messages: [{'role': 'user'/'assistant'/'system', 'content': ...}, ...]
        # → 사내 모델 호출 후 답변 문자열만 반환
        ...

start_brain_server(brain=ChatBrain(adapter=MyLocalLLM()))
```

> HTTP 로 사내 추론 서버를 호출한다면 `requests`/`httpx` 대신 표준 `urllib.request` 를 쓰세요.